# Data Governance notebook

Ez a notebook PII-osztalyozast, policy-gondolkodast es audit blueprintet mutat a WebShop Pro adatokon. A teljes lab profil:

```bash
docker compose --profile governance up -d
```

In [ ]:
import json
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'courses.json').exists())
orders = json.loads((ROOT / 'webshop-pro' / 'fixtures' / 'orders.json').read_text(encoding='utf-8'))
events = json.loads((ROOT / 'webshop-pro' / 'fixtures' / 'events.json').read_text(encoding='utf-8'))
len(orders), len(events), orders[0]

In [ ]:
classification = {
    'order_id': 'internal',
    'customer_id': 'confidential_pii',
    'city': 'confidential_pii',
    'gross_amount': 'internal_financial',
    'channel': 'internal',
    'status': 'internal',
}

def classify_record(record):
    return {field: classification.get(field, 'unclassified') for field in record}

classify_record(orders[0])

In [ ]:
def mask_order(order, role):
    masked = dict(order)
    if role not in {'support_agent', 'data_steward'}:
        masked['customer_id'] = '***'
        masked['city'] = '***'
    if role == 'marketing_analyst':
        masked['gross_amount'] = round(masked['gross_amount'] / 10000) * 10000
    return masked

for role in ['marketing_analyst', 'support_agent', 'external_partner']:
    print(role, mask_order(orders[0], role))

In [ ]:
audit_policy = [
    {'asset': 'raw.orders', 'control': 'pii_masking', 'owner': 'data-governance', 'evidence': 'policy-as-code'},
    {'asset': 'analytics.mart_daily_revenue', 'control': 'freshness_sla', 'owner': 'sales-analytics', 'evidence': 'dbt test + dashboard'},
    {'asset': 'chroma.webshop_course_materials', 'control': 'source_lineage', 'owner': 'ai-platform', 'evidence': 'document metadata'},
]
audit_policy

Kovetkezo lepes: a fenti policy-ket OPA/Rego, Unity Catalog tag vagy dbt test formajaban kodositani.